# Fine-Tuning GPT-2 for Text Generation
This notebook demonstrates how to fine-tune GPT-2 on a text dataset using Hugging Face Transformers.

In [ ]:
# Install dependencies
!pip install transformers datasets evaluate accelerate -q

In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
import os

## Load and Prepare the Dataset

In [ ]:
# Load dataset (you can replace this with your own text dataset)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train_texts = dataset['train']
eval_texts = dataset['validation']

## Load GPT-2 and Tokenizer

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token
model = GPT2LMHeadModel.from_pretrained("gpt2")

## Tokenize the Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_texts.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_texts.map(tokenize_function, batched=True, remove_columns=["text"])

## Setup Data Collator

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Define Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=10,
    push_to_hub=False
)

## Initialize Trainer and Train the Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

## Save the Model

In [ ]:
trainer.save_model("./gpt2-finetuned")

## Test the Fine-Tuned Model with a Prompt

In [ ]:
def generate_text(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_length, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_text("Once upon a time in a world ruled by AI,"))

In [ ]:
# ============================================================
# GPT-2 Fine-Tuning for Text Generation
# Dataset: WikiText-2
# ============================================================

# Install required packages (Run once in Colab/Jupyter)
# !pip install -q -U transformers datasets accelerate evaluate torch huggingface_hub

import torch
from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

# ============================================================
# 1. Load Dataset
# ============================================================

print("Loading dataset...")

# NOTE: The old bare "wikitext" repo id no longer resolves correctly with
# recent huggingface_hub/datasets versions (it expects "namespace/name").
# The dataset is now hosted under the Salesforce namespace on the Hub.
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

# ============================================================
# 2. Load GPT-2 Tokenizer and Model
# ============================================================

print("Loading GPT-2 model...")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 has no pad token, use EOS token
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")

# Resize embeddings after adding pad token
model.resize_token_embeddings(len(tokenizer))

# ============================================================
# 3. Tokenization Function
# ============================================================

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # Labels required for causal language modeling
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

print("Tokenizing dataset...")

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# ============================================================
# 4. Data Collator
# ============================================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# ============================================================
# 5. Training Arguments
# ============================================================

training_args = TrainingArguments(
    output_dir="./gpt2_finetuned",
    eval_strategy="epoch",          # Updated argument name
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,             # reduced from 3 -> much faster
    fp16=torch.cuda.is_available(), # mixed precision speedup on GPU
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

# ============================================================
# 6. Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,  # replaces deprecated tokenizer argument
    data_collator=data_collator
)

# ============================================================
# 7. Train Model
# ============================================================

print("Starting training...")

trainer.train()

# ============================================================
# 8. Save Model
# ============================================================

save_path = "./gpt2_finetuned"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")

# ============================================================
# 9. Text Generation Function
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def generate_text(prompt, max_new_tokens=100):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return generated_text

# ============================================================
# 10. Test Generation
# ============================================================

prompt = "Once upon a time in a world ruled by AI,"
generated = generate_text(prompt)

print("\nPrompt:")
print(prompt)

print("\nGenerated Text:")
print(generated)

Loading dataset...
Loading GPT-2 model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Tokenizing dataset...


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting training...


Epoch,Training Loss,Validation Loss
1,3.325811,3.317232


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./gpt2_finetuned

Prompt:
Once upon a time in a world ruled by AI,

Generated Text:
Once upon a time in a world ruled by AI, human beings have become increasingly aware of their surroundings , and have begun to do more things to ensure that human beings are protected from harm , instead of fearing a sudden invasion by AI . This may be due to their limited ability to reason , which can be inhibited by the ability to reason on their own . 

In the 1990s , the AI developed artificial intelligence ( AI ) to help them with their business problems . In 2001 , AI was officially recognized as a major threat to humanity . 

